# Collection search validation
`collect search` calls `search_candidates`; recorded fixture responses stand in for the live providers.

In [ ]:
import json, shutil, sys, tempfile
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.tests.test_collect import MINI, body, payload
from knowledge_palace.tests.test_acquisition_providers import RoutedStub
from knowledge_palace.expansion.collect import search_candidates
from knowledge_palace.graph.identity import load_vault_identity
from knowledge_palace.metadata.providers import CrossrefProvider, OpenAlexProvider
identity = load_vault_identity(MINI)
openalex_fetch = RoutedStub({"api.openalex.org": body({"results": [payload("openalex-work.json"), payload("openalex-citing.json")["results"][0]]})})
crossref_fetch = RoutedStub({"api.crossref.org": body({"message": {"items": [payload("crossref-work.json")["message"]]}})})
providers = [OpenAlexProvider(fetch=openalex_fetch), CrossrefProvider(fetch=crossref_fetch)]

In [ ]:
result = search_candidates(providers, "echo cancellation & noise 回声", 10, identity)
openalex_fetch.calls, crossref_fetch.calls

In [ ]:
[(row["key"], row["providers"], row["in_library"], row["title"]) for row in result["results"]]

In [ ]:
assert result["results"][0]["in_library"] == "alpha-2020-echo"
assert result["results"][0]["providers"] == ["openalex", "crossref"]
result["warnings"]